# 02. 설정 → TodaysMarket: 구축을 한 줄씩

| 단계 | QuantTool 코드 | ORE 원본 |
|---|---|---|
| 레시피 읽기 | `src/quanttool/data/configuration/`, `configs/` | `OREData/ored/configuration/`, `todaysmarket.xml`, `curveconfig.xml`, `conventions.xml` |
| 시세 읽기 | `src/quanttool/data/marketdata/loader.py` `Loader`, `MarketDatum`, `QuoteStore` | `OREData/ored/marketdata/loader.hpp`, `marketdatum.hpp`, `csvloader.hpp` |
| 빌더 찾기 | `src/quanttool/data/utilities/registry.py` `BUILDERS` | (C++ 클래스 계층) |
| 순서 정하기 | `src/quanttool/data/marketdata/dependencygraph.py` | `OREData/ored/marketdata/dependencygraph.hpp` |
| 보관소 | `src/quanttool/data/marketdata/market.py` `Market`, `MarketImpl` | `market.hpp`, `marketimpl.hpp` |
| 구축 | `src/quanttool/data/marketdata/todaysmarket.py` `TodaysMarket` | `todaysmarket.hpp` |

ORE 경로는 `references/Engine/` 기준. > 데이터: `tests/fixtures/ust_par_sample.csv`는 재무부 CSV와 **형식만 같은 합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE_CSV = ROOT / "tests" / "fixtures" / "ust_par_sample.csv"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.data.configuration import load_config
from quanttool.data.marketdata import QuoteStore, TodaysMarket, UstParCsvLoader
from quanttool.data.marketdata.dependencygraph import DependencyGraph
from quanttool.data.utilities.registry import BUILDERS

ASOF = date(2026, 10, 23)

## 1. 레시피 (무엇을 어떻게 만들지)
- `configs/market/todaysmarket.yaml` → `TodaysMarketParameters`: 만들 시장 객체 목록
- `configs/market/curves/ust.yaml` → `YieldCurveConfig`: 어떤 시세로, 어떤 헬퍼로, 어떤 보간으로
- `configs/conventions/rates.yaml` → `RateConventions`: 결제일, 일수, 쿠폰 주기

In [ ]:
cfg = load_config(ROOT / "configs")
print("구축 대상:", cfg.todaysmarket.yield_curves)
cfg.curve_configs["UST"].model_dump()

In [ ]:
cfg.conventions["UST"].model_dump()

## 2. 시세 읽기 → MarketDatum → QuoteStore
Loader는 그 날짜의 시세를 `MarketDatum(asof, id, value)` 목록으로 준다(ORE `market.txt`의 `날짜 키 값`과 같은 모양). QuoteStore는 id마다 `SimpleQuote`를 하나 만든다.

In [ ]:
loader = UstParCsvLoader(SAMPLE_CSV)
quotes = loader.load(ASOF)
pd.DataFrame([q.__dict__ for q in quotes])

In [ ]:
store = QuoteStore(quotes)
store["UST/PAR/10Y"], store["UST/PAR/10Y"].value()

## 3. 빌더 찾기와 구축 순서
레시피의 `type`(`YieldCurve`)으로 등록부에서 빌더를 찾는다. 커브가 서로를 참조하면(예: 스왑 커브 → 할인 커브) 의존 그래프가 순서를 정한다. 지금은 UST 하나뿐이다.

In [ ]:
print("등록된 빌더:", BUILDERS.keys())
builder = BUILDERS["YieldCurve"](cfg.conventions)
recipe = cfg.curve_configs["UST"]
graph = DependencyGraph({("YieldCurve", "UST"): builder.requires(recipe)})
graph.order([("YieldCurve", "UST")])

## 4. TodaysMarket 생성자가 하는 일
`todaysmarket.py`를 열어 보면 생성자가 아래를 한다.
1. `MarketImpl.__init__`: 날짜, QuoteStore, 빈 보관소
2. 평가일을 asof로 두고(헬퍼 결제일 계산이 평가일을 쓰므로) 의존 순서대로 빌더 실행
3. 커브를 바로 부트스트랩하고 **얼린다**(`freeze`) — 이후 시세·평가일이 바뀌어도 그대로
4. `RelinkableYieldTermStructureHandle`에 넣어 보관소에 이름으로 등록

In [ ]:
market = TodaysMarket(ASOF, cfg.todaysmarket, loader, cfg.curve_configs, cfg.conventions)
# 같은 것: TodaysMarket.from_config(ASOF, cfg, loader)
print("보관된 커브", market.yield_curve_names(), " 헬퍼 수", len(market.curve_helpers("UST")))
print("기준일", market.yield_curve("UST").referenceDate())

## 5. 시장에서 이름으로 꺼내 쓰기
가격결정 코드는 시장에서 **이름으로** 커브를 꺼낸다(원칙 3). 커브는 아무 날짜에 답하는 함수다.

In [ ]:
curve = market.yield_curve("UST")
d = ql.Date(17, 3, 2031)
print("DF", curve.discount(d))
print("zero", curve.zeroRate(d, ql.Actual365Fixed(), ql.Continuous).rate())
print(
    "1Y fwd from 2031-03-17",
    curve.forwardRate(d, d + ql.Period("1Y"), ql.Actual365Fixed(), ql.Continuous).rate(),
)

## 6. 진단과 커브 표
`src/quanttool/data/marketdata/diagnostics.py`, `src/quanttool/analytics/app/marketmonitor.py`

In [ ]:
from quanttool.analytics.app import curve_shape, curve_table
from quanttool.data.marketdata.diagnostics import helper_repricing

helper_repricing(market, "UST")

In [ ]:
table = curve_table(market, "UST", cfg)
display((table[["par", "zero_cc", "fwd_1y"]] * 100).round(3))
curve_shape(table)

## 7. 얼린 시장은 다른 날짜 시장과 독립이다
날짜가 다른 시장을 여러 개 만들어도 각 시장의 커브는 자기 asof에 고정되어 있다. 다만 **상품 평가**는 전역 평가일을 쓰므로 각 시장의 `active()` 안에서 한다(03번).

In [ ]:
prev = TodaysMarket.from_config(date(2026, 10, 22), cfg, loader)
for m in (prev, market):
    c = m.yield_curve("UST")
    print(
        m.asof,
        "기준일",
        c.referenceDate(),
        " 10Y zero",
        round(c.zeroRate(10.0, ql.Continuous).rate() * 100, 4),
    )

다음: `03_trade_to_npv.ipynb` — 계약서를 이 시장에 붙여 가격을 낸다.